# Deploy DynamoDB Tables for Betting & Gaming Platform

Creates DynamoDB tables and seeds demo data for **both** use cases:
1. **Smart Betting Assistant** — sports, events, odds, bet placement
2. **Customer Support Agent** — customer profiles, transactions, support tickets

**Tables:**
| Table | PK | SK | GSI | Purpose |
|-------|----|----|-----|---------|
| `BettingSports` | `id` | — | — | Available sports |
| `BettingEvents` | `sportId` | `id` | `StatusIndex` (status) | Sporting events |
| `BettingOdds` | `eventId` | — | — | Odds per event |
| `BettingBets` | `betId` | — | `AccountIndex` (accountId) | Placed bets |
| `BettingAccounts` | `accountId` | — | `EmailIndex` (email) | Customer profiles + balances |
| `BettingPromotions` | `id` | — | — | Promotional offers |
| `BettingTransactions` | `accountId` | `transactionId` | — | Deposits, withdrawals, settlements |
| `BettingSupportTickets` | `ticketId` | — | `AccountIndex` (accountId) | Support cases |

<hr>

In [1]:
import boto3
import json
import time
from decimal import Decimal

REGION = "us-east-1"
dynamodb = boto3.resource("dynamodb", region_name=REGION)
ddb_client = boto3.client("dynamodb", region_name=REGION)

TABLE_PREFIX = "Betting"

TABLE_DEFINITIONS = [
    {
        "TableName": f"{TABLE_PREFIX}Sports",
        "KeySchema": [{"AttributeName": "id", "KeyType": "HASH"}],
        "AttributeDefinitions": [{"AttributeName": "id", "AttributeType": "S"}],
    },
    {
        "TableName": f"{TABLE_PREFIX}Events",
        "KeySchema": [
            {"AttributeName": "sportId", "KeyType": "HASH"},
            {"AttributeName": "id", "KeyType": "RANGE"},
        ],
        "AttributeDefinitions": [
            {"AttributeName": "sportId", "AttributeType": "S"},
            {"AttributeName": "id", "AttributeType": "S"},
            {"AttributeName": "status", "AttributeType": "S"},
        ],
        "GlobalSecondaryIndexes": [
            {
                "IndexName": "StatusIndex",
                "KeySchema": [
                    {"AttributeName": "status", "KeyType": "HASH"},
                    {"AttributeName": "id", "KeyType": "RANGE"},
                ],
                "Projection": {"ProjectionType": "ALL"},
            }
        ],
    },
    {
        "TableName": f"{TABLE_PREFIX}Odds",
        "KeySchema": [{"AttributeName": "eventId", "KeyType": "HASH"}],
        "AttributeDefinitions": [{"AttributeName": "eventId", "AttributeType": "S"}],
    },
    {
        "TableName": f"{TABLE_PREFIX}Bets",
        "KeySchema": [{"AttributeName": "betId", "KeyType": "HASH"}],
        "AttributeDefinitions": [
            {"AttributeName": "betId", "AttributeType": "S"},
            {"AttributeName": "accountId", "AttributeType": "S"},
        ],
        "GlobalSecondaryIndexes": [
            {
                "IndexName": "AccountIndex",
                "KeySchema": [
                    {"AttributeName": "accountId", "KeyType": "HASH"},
                    {"AttributeName": "betId", "KeyType": "RANGE"},
                ],
                "Projection": {"ProjectionType": "ALL"},
            }
        ],
    },
    {
        "TableName": f"{TABLE_PREFIX}Accounts",
        "KeySchema": [{"AttributeName": "accountId", "KeyType": "HASH"}],
        "AttributeDefinitions": [
            {"AttributeName": "accountId", "AttributeType": "S"},
            {"AttributeName": "email", "AttributeType": "S"},
        ],
        "GlobalSecondaryIndexes": [
            {
                "IndexName": "EmailIndex",
                "KeySchema": [
                    {"AttributeName": "email", "KeyType": "HASH"},
                ],
                "Projection": {"ProjectionType": "ALL"},
            }
        ],
    },
    {
        "TableName": f"{TABLE_PREFIX}Promotions",
        "KeySchema": [{"AttributeName": "id", "KeyType": "HASH"}],
        "AttributeDefinitions": [{"AttributeName": "id", "AttributeType": "S"}],
    },
    {
        "TableName": f"{TABLE_PREFIX}Transactions",
        "KeySchema": [
            {"AttributeName": "accountId", "KeyType": "HASH"},
            {"AttributeName": "transactionId", "KeyType": "RANGE"},
        ],
        "AttributeDefinitions": [
            {"AttributeName": "accountId", "AttributeType": "S"},
            {"AttributeName": "transactionId", "AttributeType": "S"},
        ],
    },
    {
        "TableName": f"{TABLE_PREFIX}SupportTickets",
        "KeySchema": [{"AttributeName": "ticketId", "KeyType": "HASH"}],
        "AttributeDefinitions": [
            {"AttributeName": "ticketId", "AttributeType": "S"},
            {"AttributeName": "accountId", "AttributeType": "S"},
        ],
        "GlobalSecondaryIndexes": [
            {
                "IndexName": "AccountIndex",
                "KeySchema": [
                    {"AttributeName": "accountId", "KeyType": "HASH"},
                    {"AttributeName": "ticketId", "KeyType": "RANGE"},
                ],
                "Projection": {"ProjectionType": "ALL"},
            }
        ],
    },
]

print(f"Defined {len(TABLE_DEFINITIONS)} tables")

Defined 8 tables


In [2]:
for table_def in TABLE_DEFINITIONS:
    table_name = table_def["TableName"]
    try:
        ddb_client.describe_table(TableName=table_name)
        print(f"  Deleting existing table: {table_name}")
        ddb_client.delete_table(TableName=table_name)
        ddb_client.get_waiter("table_not_exists").wait(TableName=table_name)
    except ddb_client.exceptions.ResourceNotFoundException:
        pass

    create_params = {
        "TableName": table_name,
        "KeySchema": table_def["KeySchema"],
        "AttributeDefinitions": table_def["AttributeDefinitions"],
        "BillingMode": "PAY_PER_REQUEST",
    }
    if "GlobalSecondaryIndexes" in table_def:
        create_params["GlobalSecondaryIndexes"] = table_def["GlobalSecondaryIndexes"]

    ddb_client.create_table(**create_params)
    print(f"  Creating table: {table_name}")

print("\nWaiting for tables to become ACTIVE...")
for table_def in TABLE_DEFINITIONS:
    table_name = table_def["TableName"]
    ddb_client.get_waiter("table_exists").wait(TableName=table_name)
    print(f"  OK {table_name}")

print("\nAll tables created!")

  Creating table: BettingSports
  Creating table: BettingEvents
  Creating table: BettingOdds
  Creating table: BettingBets
  Creating table: BettingAccounts
  Creating table: BettingPromotions
  Creating table: BettingTransactions
  Creating table: BettingSupportTickets

Waiting for tables to become ACTIVE...
  OK BettingSports
  OK BettingEvents
  OK BettingOdds
  OK BettingBets
  OK BettingAccounts
  OK BettingPromotions
  OK BettingTransactions
  OK BettingSupportTickets

All tables created!


## Seed Data\n<hr>

In [3]:
sports_table = dynamodb.Table(f"{TABLE_PREFIX}Sports")

SPORTS = [
    {"id": "football",     "name": "Football",     "activeEvents": 24},
    {"id": "basketball",   "name": "Basketball",   "activeEvents": 18},
    {"id": "tennis",       "name": "Tennis",       "activeEvents": 32},
    {"id": "horse_racing", "name": "Horse Racing", "activeEvents": 12},
    {"id": "esports",      "name": "Esports",      "activeEvents": 8},
]

with sports_table.batch_writer() as batch:
    for sport in SPORTS:
        batch.put_item(Item=sport)

print(f"Seeded {len(SPORTS)} sports")

Seeded 5 sports


In [4]:
events_table = dynamodb.Table(f"{TABLE_PREFIX}Events")

EVENTS = [
    {"sportId": "football", "id": "evt-001", "home": "Arsenal", "away": "Chelsea", "league": "Premier League", "kickoff": "2026-04-25T15:00:00Z", "status": "upcoming"},
    {"sportId": "football", "id": "evt-002", "home": "Barcelona", "away": "Real Madrid", "league": "La Liga", "kickoff": "2026-04-25T20:00:00Z", "status": "upcoming"},
    {"sportId": "football", "id": "evt-003", "home": "Bayern Munich", "away": "Dortmund", "league": "Bundesliga", "kickoff": "2026-04-26T17:30:00Z", "status": "upcoming"},
    {"sportId": "basketball", "id": "evt-010", "home": "Lakers", "away": "Celtics", "league": "NBA", "kickoff": "2026-04-25T19:30:00Z", "status": "live"},
    {"sportId": "basketball", "id": "evt-011", "home": "Warriors", "away": "Bucks", "league": "NBA", "kickoff": "2026-04-26T21:00:00Z", "status": "upcoming"},
    {"sportId": "tennis", "id": "evt-020", "home": "Djokovic", "away": "Alcaraz", "league": "ATP Masters", "kickoff": "2026-04-25T14:00:00Z", "status": "live"},
    {"sportId": "tennis", "id": "evt-021", "home": "Sinner", "away": "Medvedev", "league": "ATP Masters", "kickoff": "2026-04-26T16:00:00Z", "status": "upcoming"},
    {"sportId": "horse_racing", "id": "evt-030", "home": "Ascot - 3:15 Race", "away": "8 runners", "league": "Royal Ascot", "kickoff": "2026-04-25T15:15:00Z", "status": "upcoming"},
    {"sportId": "esports", "id": "evt-040", "home": "T1", "away": "Gen.G", "league": "LCK", "kickoff": "2026-04-25T12:00:00Z", "status": "live"},
]

with events_table.batch_writer() as batch:
    for event in EVENTS:
        batch.put_item(Item=event)

print(f"Seeded {len(EVENTS)} events")

Seeded 9 events


In [5]:
odds_table = dynamodb.Table(f"{TABLE_PREFIX}Odds")

ODDS = [
    {"eventId": "evt-001", "odds": {"homeWin": Decimal("2.10"), "draw": Decimal("3.40"), "awayWin": Decimal("3.20")}},
    {"eventId": "evt-002", "odds": {"homeWin": Decimal("1.85"), "draw": Decimal("3.60"), "awayWin": Decimal("4.00")}},
    {"eventId": "evt-003", "odds": {"homeWin": Decimal("1.70"), "draw": Decimal("3.80"), "awayWin": Decimal("4.50")}},
    {"eventId": "evt-010", "odds": {"homeWin": Decimal("1.90"), "awayWin": Decimal("1.95"), "overTotal": Decimal("1.85"), "underTotal": Decimal("2.00")}},
    {"eventId": "evt-011", "odds": {"homeWin": Decimal("2.20"), "awayWin": Decimal("1.70"), "overTotal": Decimal("1.90"), "underTotal": Decimal("1.95")}},
    {"eventId": "evt-020", "odds": {"homeWin": Decimal("1.60"), "awayWin": Decimal("2.30")}},
    {"eventId": "evt-021", "odds": {"homeWin": Decimal("1.75"), "awayWin": Decimal("2.10")}},
    {"eventId": "evt-030", "odds": {"runner1": Decimal("3.50"), "runner2": Decimal("5.00"), "runner3": Decimal("7.00"), "runner4": Decimal("8.50"), "runner5": Decimal("12.00")}},
    {"eventId": "evt-040", "odds": {"homeWin": Decimal("1.45"), "awayWin": Decimal("2.70")}},
]

with odds_table.batch_writer() as batch:
    for item in ODDS:
        batch.put_item(Item=item)

print(f"Seeded {len(ODDS)} odds records")

Seeded 9 odds records


In [6]:
promotions_table = dynamodb.Table(f"{TABLE_PREFIX}Promotions")

PROMOTIONS = [
    {"id": "promo-001", "name": "Welcome Bonus", "description": "100% match on first deposit up to $200", "minDeposit": 10, "expires": "2026-05-31"},
    {"id": "promo-002", "name": "Free Bet Friday", "description": "Place 3 bets on Friday, get a $10 free bet", "minOdds": Decimal("1.50"), "expires": "2026-04-30"},
    {"id": "promo-003", "name": "Acca Insurance", "description": "5+ leg accumulator insurance - stake back if one leg loses", "minLegs": 5, "expires": "2026-06-30"},
    {"id": "promo-004", "name": "Enhanced Odds", "description": "Arsenal to win at 5.00 (was 2.10) - max stake $20", "eventId": "evt-001", "expires": "2026-04-25"},
]

with promotions_table.batch_writer() as batch:
    for promo in PROMOTIONS:
        batch.put_item(Item=promo)

print(f"Seeded {len(PROMOTIONS)} promotions")

Seeded 4 promotions


In [7]:
accounts_table = dynamodb.Table(f"{TABLE_PREFIX}Accounts")

ACCOUNTS = [
    {
        "accountId": "default",
        "email": "john.doe@example.com",
        "name": "John Doe",
        "balance": Decimal("1000.00"),
        "currency": "USD",
        "tier": "gold",
        "kycStatus": "verified",
        "registeredAt": "2025-11-15T09:00:00Z",
        "lastLoginAt": "2026-04-23T08:30:00Z",
        "depositLimit": Decimal("5000.00"),
        "weeklyBetLimit": Decimal("2000.00"),
        "selfExcluded": False,
    },
    {
        "accountId": "cust-002",
        "email": "maria.garcia@example.com",
        "name": "Maria Garcia",
        "balance": Decimal("250.00"),
        "currency": "USD",
        "tier": "silver",
        "kycStatus": "verified",
        "registeredAt": "2026-01-20T14:00:00Z",
        "lastLoginAt": "2026-04-22T19:15:00Z",
        "depositLimit": Decimal("2000.00"),
        "weeklyBetLimit": Decimal("1000.00"),
        "selfExcluded": False,
    },
    {
        "accountId": "cust-003",
        "email": "alex.new@example.com",
        "name": "Alex Thompson",
        "balance": Decimal("200.00"),
        "currency": "USD",
        "tier": "bronze",
        "kycStatus": "pending",
        "registeredAt": "2026-04-22T10:00:00Z",
        "lastLoginAt": "2026-04-23T07:45:00Z",
        "depositLimit": Decimal("1000.00"),
        "weeklyBetLimit": Decimal("500.00"),
        "selfExcluded": False,
    },
]

with accounts_table.batch_writer() as batch:
    for acct in ACCOUNTS:
        batch.put_item(Item=acct)

print(f"Seeded {len(ACCOUNTS)} customer accounts")

Seeded 3 customer accounts


In [8]:
transactions_table = dynamodb.Table(f"{TABLE_PREFIX}Transactions")

TRANSACTIONS = [
    # John Doe (default)
    {"accountId": "default", "transactionId": "txn-001", "type": "deposit",    "amount": Decimal("500.00"),  "method": "credit_card", "status": "completed", "createdAt": "2025-11-15T09:05:00Z", "description": "Initial deposit"},
    {"accountId": "default", "transactionId": "txn-002", "type": "deposit",    "amount": Decimal("300.00"),  "method": "bank_transfer", "status": "completed", "createdAt": "2026-01-10T12:00:00Z", "description": "Bank transfer deposit"},
    {"accountId": "default", "transactionId": "txn-003", "type": "bonus",      "amount": Decimal("200.00"),  "method": "system", "status": "completed", "createdAt": "2025-11-15T09:06:00Z", "description": "Welcome bonus credited"},
    {"accountId": "default", "transactionId": "txn-004", "type": "withdrawal", "amount": Decimal("-150.00"), "method": "bank_transfer", "status": "completed", "createdAt": "2026-03-01T10:00:00Z", "description": "Withdrawal to bank"},
    {"accountId": "default", "transactionId": "txn-005", "type": "bet_settlement", "amount": Decimal("150.00"), "method": "system", "status": "completed", "createdAt": "2026-04-20T18:00:00Z", "description": "Winning bet payout - evt-020"},
    # Maria Garcia
    {"accountId": "cust-002", "transactionId": "txn-010", "type": "deposit",    "amount": Decimal("200.00"),  "method": "credit_card", "status": "completed", "createdAt": "2026-01-20T14:05:00Z", "description": "Initial deposit"},
    {"accountId": "cust-002", "transactionId": "txn-011", "type": "deposit",    "amount": Decimal("100.00"),  "method": "credit_card", "status": "completed", "createdAt": "2026-04-18T09:00:00Z", "description": "Top-up deposit"},
    {"accountId": "cust-002", "transactionId": "txn-012", "type": "bet_settlement", "amount": Decimal("-50.00"), "method": "system", "status": "completed", "createdAt": "2026-04-19T20:00:00Z", "description": "Lost bet - evt-003"},
    # Alex Thompson (new customer - support scenario)
    {"accountId": "cust-003", "transactionId": "txn-020", "type": "deposit",    "amount": Decimal("200.00"),  "method": "credit_card", "status": "completed", "createdAt": "2026-04-22T10:05:00Z", "description": "Initial deposit"},
]

with transactions_table.batch_writer() as batch:
    for txn in TRANSACTIONS:
        batch.put_item(Item=txn)

print(f"Seeded {len(TRANSACTIONS)} transactions")

Seeded 9 transactions


In [9]:
tickets_table = dynamodb.Table(f"{TABLE_PREFIX}SupportTickets")

TICKETS = [
    {
        "ticketId": "tkt-001",
        "accountId": "cust-002",
        "subject": "Free Bet Friday not received",
        "description": "I placed 3 bets last Friday but did not receive my free bet.",
        "status": "open",
        "priority": "medium",
        "category": "promotions",
        "createdAt": "2026-04-21T10:00:00Z",
        "updatedAt": "2026-04-21T10:00:00Z",
    },
    {
        "ticketId": "tkt-002",
        "accountId": "cust-003",
        "subject": "Welcome bonus not applied",
        "description": "I deposited $200 yesterday but the welcome bonus has not appeared in my account.",
        "status": "open",
        "priority": "high",
        "category": "bonus",
        "createdAt": "2026-04-23T08:00:00Z",
        "updatedAt": "2026-04-23T08:00:00Z",
    },
    {
        "ticketId": "tkt-003",
        "accountId": "default",
        "subject": "Wrong bet selection",
        "description": "I accidentally bet on the wrong selection for Arsenal vs Chelsea. Can I change it?",
        "status": "open",
        "priority": "medium",
        "category": "betting",
        "createdAt": "2026-04-23T09:30:00Z",
        "updatedAt": "2026-04-23T09:30:00Z",
    },
]

with tickets_table.batch_writer() as batch:
    for ticket in TICKETS:
        batch.put_item(Item=ticket)

print(f"Seeded {len(TICKETS)} support tickets")

Seeded 3 support tickets


In [10]:
print("BettingBets table created empty — bets are persisted at runtime.")
print("BettingBets now includes accountId field + AccountIndex GSI for customer lookups.")

BettingBets table created empty — bets are persisted at runtime.
BettingBets now includes accountId field + AccountIndex GSI for customer lookups.


## Verify Seed Data\n<hr>

In [11]:
from boto3.dynamodb.conditions import Key

print("=" * 60)
print("Verification")
print("=" * 60)

for td in TABLE_DEFINITIONS:
    t = dynamodb.Table(td["TableName"])
    count = t.scan(Select="COUNT")["Count"]
    print(f"  {td['TableName']:35s} -> {count} items")

print("\n--- Sample: Football events ---")
et = dynamodb.Table(f"{TABLE_PREFIX}Events")
for evt in et.query(KeyConditionExpression=Key("sportId").eq("football"))["Items"]:
    print(f"  {evt['id']}: {evt['home']} vs {evt['away']} ({evt['status']})")

print("\n--- Sample: Live events (GSI) ---")
for evt in et.query(IndexName="StatusIndex", KeyConditionExpression=Key("status").eq("live"))["Items"]:
    print(f"  {evt['id']}: {evt['home']} vs {evt['away']} [{evt['sportId']}]")

print("\n--- Sample: Customer accounts ---")
at = dynamodb.Table(f"{TABLE_PREFIX}Accounts")
for acct in at.scan()["Items"]:
    print(f"  {acct['accountId']}: {acct['name']} ({acct['email']}) - ${acct['balance']} - tier={acct['tier']} kyc={acct['kycStatus']}")

print("\n--- Sample: Transactions for default account ---")
tt = dynamodb.Table(f"{TABLE_PREFIX}Transactions")
for txn in tt.query(KeyConditionExpression=Key("accountId").eq("default"))["Items"]:
    print(f"  {txn['transactionId']}: {txn['type']:15s} ${txn['amount']:>8} - {txn['description']}")

print("\n--- Sample: Open support tickets ---")
st = dynamodb.Table(f"{TABLE_PREFIX}SupportTickets")
for tkt in st.scan()["Items"]:
    print(f"  {tkt['ticketId']}: [{tkt['priority']}] {tkt['subject']} (account={tkt['accountId']})")

print("\nAll tables ready for both demo scenarios!")

Verification
  BettingSports                       -> 5 items
  BettingEvents                       -> 9 items
  BettingOdds                         -> 9 items
  BettingBets                         -> 0 items
  BettingAccounts                     -> 3 items
  BettingPromotions                   -> 4 items
  BettingTransactions                 -> 9 items
  BettingSupportTickets               -> 3 items

--- Sample: Football events ---
  evt-001: Arsenal vs Chelsea (upcoming)
  evt-002: Barcelona vs Real Madrid (upcoming)
  evt-003: Bayern Munich vs Dortmund (upcoming)

--- Sample: Live events (GSI) ---
  evt-010: Lakers vs Celtics [basketball]
  evt-020: Djokovic vs Alcaraz [tennis]
  evt-040: T1 vs Gen.G [esports]

--- Sample: Customer accounts ---
  cust-002: Maria Garcia (maria.garcia@example.com) - $250 - tier=silver kyc=verified
  cust-003: Alex Thompson (alex.new@example.com) - $200 - tier=bronze kyc=pending
  default: John Doe (john.doe@example.com) - $1000 - tier=gold kyc=verifi

## Clean Up (run only when done with the demo)\n<hr>

In [ ]:
# # Uncomment to delete all tables
# for table_def in TABLE_DEFINITIONS:
#     table_name = table_def["TableName"]
#     try:
#         ddb_client.delete_table(TableName=table_name)
#         print(f"Deleting {table_name}...")
#     except ddb_client.exceptions.ResourceNotFoundException:
#         print(f"{table_name} not found, skipping.")
# print("Done.")